<a href="https://colab.research.google.com/github/hyunjin2724/code-based-document-test/blob/main/Code_to_Manual.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install python-docx if not already installed
!pip install python-docx

import os
import zipfile
import re
from docx import Document
from openai import OpenAI
from google.colab import userdata

# 1. 환경 설정 (OpenAI API 키 필요)
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)

class ManualAgentSystem:
    def __init__(self, zip_path, output_name="Manual_Result.docx"):
        self.zip_path = zip_path
        self.extract_path = "source_temp"
        self.output_name = output_name
        self.doc = Document()
        self.collected_code = ""

    def run(self):
        print("🚀 [에이전트 시스템 가동]")
        self._unzip_source()
        self._analyze_stage()
        self._write_stage()
        print(f"✅ 모든 작업 완료! 파일명: {self.output_name}")

    # [에이전트 A: 소스 분석 및 추출]
    def _unzip_source(self):
        print("📂 Stage 1: 압축 해제 및 소스 추출 중...")
        with zipfile.ZipFile(self.zip_path, 'r') as zip_ref:
            zip_ref.extractall(self.extract_path)

        # 핵심 파일(Controller, Service, Entity)만 골라 텍스트로 합침
        for root, _, files in os.walk(self.extract_path):
            for file in files:
                if file.endswith((".java", ".properties")):
                    with open(os.path.join(root, file), 'r', encoding='utf-8') as f:
                        self.collected_code += f"\n--- File: {file} ---\n" + f.read()

    # [에이전트 B: 기술 해석 (LLM)]
    def _analyze_stage(self):
        print("🧠 Stage 2: 기술 해석 에이전트가 로직 분석 중...")
        prompt = f"""
        당신은 시니어 자바 개발자이자 기술 작가입니다.
        아래 제공된 소스 코드를 분석하여 '사용자 매뉴얼'에 들어갈 핵심 내용을 정리하세요.

        [요구사항]
        - 대제목은 1, 2, 3...
        - 중제목은 a, b, c...
        - 소제목은 ㄱ, ㄴ, ㄷ... 형식을 유지할 것.
        - 사용자가 이해하기 쉬운 비즈니스 용어를 사용할 것.

        [소스 코드]
        {self.collected_code[:15000]} # 토큰 제한을 고려해 일부 절삭
        """

        response = client.chat.completions.create(
            model="gpt-4o", # 또는 gpt-3.5-turbo
            messages=[{"role": "user", "content": prompt}]
        )
        self.analysis_result = response.choices[0].message.content

    # [에이전트 C: 문서 작성 및 서식 적용]
    def _write_stage(self):
        print("✍️ Stage 3: 기술 작가 에이전트가 Word 파일 생성 중...")
        lines = self.analysis_result.split('\n')

        for line in lines:
            line = line.strip()
            if not line: continue

            # 숫자(1., 2.)로 시작하면 대제목
            if re.match(r'^\d+\.', line):
                self.doc.add_heading(line, level=1)
            # 알파벳(a., b.)으로 시작하면 중제목
            elif re.match(r'^[a-z]\.', line):
                self.doc.add_heading(line, level=2)
            # 한글 자음(ㄱ., ㄴ.)으로 시작하면 소제목
            elif re.match(r'^[ㄱ-ㅎ]\.', line):
                p = self.doc.add_paragraph(line)
                p.style = 'List Bullet'
            else:
                self.doc.add_paragraph(line)

        self.doc.save(self.output_name)

# 실행부
if __name__ == "__main__":
    # ZIP 파일 경로를 입력하세요
    agent_system = ManualAgentSystem("WedemyServer-main (1).zip")
    agent_system.run()

🚀 [에이전트 시스템 가동]
📂 Stage 1: 압축 해제 및 소스 추출 중...
🧠 Stage 2: 기술 해석 에이전트가 로직 분석 중...
✍️ Stage 3: 기술 작가 에이전트가 Word 파일 생성 중...
✅ 모든 작업 완료! 파일명: Manual_Result.docx


In [ ]:
from google.colab import files

# 파일을 선택하여 업로드합니다.
uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))


Saving WedemyServer-main.zip to WedemyServer-main (1).zip
User uploaded file "WedemyServer-main (1).zip" with length 288765 bytes


In [ ]:
import os
import zipfile
import re
from docx import Document
from openai import OpenAI
from google.colab import userdata

# 환경 설정 (본인의 API 키를 입력하세요)
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)

class AutoPromptManualSystem:
    def __init__(self, zip_path, output_name="Smart_User_Manual.docx"):
        self.zip_path = zip_path
        self.extract_path = "source_temp"
        self.output_name = output_name
        self.doc = Document()
        self.collected_code = ""

    def run(self):
        print("🤖 [스마트 에이전트 시스템 가동]")
        self._unzip_source()

        # 1단계: 코드를 분석하여 '사용자 메뉴 가이드라인' 생성
        meta_instruction = self._generate_meta_instruction()

        # 2단계: 생성된 가이드라인을 바탕으로 최종 매뉴얼 작성
        self._write_manual(meta_instruction)
        print(f"✅ 작업 완료! 파일명: {self.output_name}")

    def _unzip_source(self):
        print("📂 Stage 1: 소스 코드 수집 중...")
        with zipfile.ZipFile(self.zip_path, 'r') as zip_ref:
            zip_ref.extractall(self.extract_path)
        for root, _, files in os.walk(self.extract_path):
            for file in files:
                if file.endswith((".java", ".properties")):
                    with open(os.path.join(root, file), 'r', encoding='utf-8') as f:
                        self.collected_code += f"\n--- File: {file} ---\n" + f.read()

    def _generate_meta_instruction(self):
        print("🧠 Stage 2: [분석 에이전트] 코드를 기반으로 메뉴 구조 파악 중...")
        # AI에게 먼저 코드를 보고 '사용자용 메뉴판'을 만들라고 시킵니다.
        analysis_prompt = f"""
        아래 자바 소스 코드를 분석해서 일반 사용자가 보게 될 '웹사이트 메뉴 구조'를 추론해내세요.
        각 API 경로와 로직이 실제 화면에서 어떤 서비스 메뉴로 보일지 리스트업하세요.

        [코드 데이터]
        {self.collected_code[:12000]}
        """

        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "system", "content": "너는 전문 비즈니스 분석가야."},
                      {"role": "user", "content": analysis_prompt}]
        )
        menu_structure = response.choices[0].message.content

        # 분석된 메뉴 구조를 바탕으로 작가에게 줄 지침(프롬프트)을 생성합니다.
        print("📝 Stage 3: [프롬프트 에이전트] 최적의 작가 지침 생성 완료.")
        return f"""
        너는 전문 기술 작가야. 아래의 메뉴 구조를 기반으로 사용자 매뉴얼을 작성해.

        [분석된 메뉴 구조]
        {menu_structure}

        [작성 규칙]
        1. 대제목(1, 2), 중제목(a, b), 소제목(ㄱ, ㄴ) 형식을 반드시 지킬 것.
        2. 개발 용어 대신 사용자 중심의 쉬운 단어를 사용할 것.
        """

    def _write_manual(self, final_instruction):
        print("✍️ Stage 4: [작가 에이전트] 최종 매뉴얼 작성 및 Word 변환 중...")
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": final_instruction}]
        )
        content = response.choices[0].message.content

        # Word 파일 저장 로직 (스타일 적용)
        for line in content.split('\n'):
            line = line.strip()
            if not line: continue
            if re.match(r'^\d+\.', line): self.doc.add_heading(line, level=1)
            elif re.match(r'^[a-z]\.', line): self.doc.add_heading(line, level=2)
            elif re.match(r'^[ㄱ-ㅎ]\.', line): self.doc.add_paragraph(line, style='List Bullet')
            else: self.doc.add_paragraph(line)

        self.doc.save(self.output_name)

# 실행 (ZIP 파일 이름만 맞춰주세요)
if __name__ == "__main__":
    agent = AutoPromptManualSystem("WedemyServer-main.zip")
    agent.run()

🤖 [스마트 에이전트 시스템 가동]
📂 Stage 1: 소스 코드 수집 중...
🧠 Stage 2: [분석 에이전트] 코드를 기반으로 메뉴 구조 파악 중...
📝 Stage 3: [프롬프트 에이전트] 최적의 작가 지침 생성 완료.
✍️ Stage 4: [작가 에이전트] 최종 매뉴얼 작성 및 Word 변환 중...
✅ 작업 완료! 파일명: Smart_User_Manual.docx
